# Pipeline 3 : Qu'est-ce qui distingue une molecule active ?

La question du chercheur : avant de lancer des modeles, que nous disent deja les donnees sur les molecules qui marchent contre l'EGFR ?

Cette etape d'exploration est souvent la plus riche en enseignements. On compare les proprietes des molecules actives et inactives, on cherche les differences systematiques, et on essaie de degager une premiere intuition chimique. Si une propriete separe clairement les deux groupes, elle sera precieuse pour les modeles, et surtout elle donne au chercheur une regle simple a garder en tete.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

C_BLEU = "#1f6f8b"
C_ORANGE = "#e0771a"
C_VERT = "#2e8b57"
C_ROUGE = "#9b2226"
C_GRIS = "#8d99ae"

plt.rcParams.update({
    "figure.figsize": (13, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13
})

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('egfr_descripteurs.csv')
# On recree l'etiquette d'activite a partir du pIC50
df['activite'] = np.where(df['pIC50'] >= 6, 'actif',
                          np.where(df['pIC50'] < 5, 'inactif', 'intermediaire'))
print(f"Molecules : {len(df)}")
print(df['activite'].value_counts())

# 1. La distribution de l'activite

In [ ]:
plt.figure(figsize=(13, 5))
sns.histplot(df['pIC50'], bins=50, color=C_BLEU, kde=True)
plt.axvline(5, color=C_GRIS, linestyle='--', linewidth=1.3, label='Seuil inactif')
plt.axvline(6, color=C_ORANGE, linestyle='--', linewidth=1.3, label='Seuil actif')
plt.title("Distribution de la puissance d'inhibition (pIC50) sur l'EGFR")
plt.xlabel("pIC50 (plus haut = plus puissant)")
plt.ylabel("Nombre de molecules")
plt.legend()
plt.tight_layout()
plt.show()

print("La distribution du pIC50 s'etale surtout entre 4 et 9, avec une concentration dans la zone active au-dela de 6. Cela reflete un biais naturel des bases de donnees : les chercheurs publient et testent surtout des molecules qu'ils esperent actives, donc les molecules puissantes sont surrepresentees par rapport a un tirage aleatoire du monde chimique. C'est une information importante pour interpreter honnetement nos taux de reussite plus tard.")

# 2. Actives contre inactives : les proprietes comparees

On superpose les distributions de chaque descripteur pour les molecules actives et inactives. La ou les deux courbes se separent, on tient une propriete discriminante.

In [ ]:
df_bin = df[df['activite'] != 'intermediaire'].copy()
descripteurs = ['poids_moleculaire', 'logP', 'tpsa', 'donneurs_H',
                'accepteurs_H', 'liaisons_rotatives']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, desc in zip(axes.flat, descripteurs):
    for etat, couleur in [('actif', C_VERT), ('inactif', C_ROUGE)]:
        sous = df_bin[df_bin['activite'] == etat]
        sns.kdeplot(sous[desc], ax=ax, color=couleur, fill=True, alpha=0.35, label=etat)
    ax.set_title(desc)
    ax.legend(fontsize=9)

plt.suptitle("Proprietes des molecules actives (vert) contre inactives (rouge)", y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

print("Les differences sont reelles mais subtiles, et c'est un enseignement en soi. Aucun descripteur pris seul ne separe parfaitement les actives des inactives, ce qui signifie qu'aucune regle simple du type molecule lourde egale molecule active ne fonctionne. L'activite resulte d'une combinaison de proprietes, et c'est precisement pour ca qu'on a besoin de modeles de machine learning capables de croiser plusieurs dimensions plutot que d'un seuil unique.")

# 3. Correlations entre descripteurs

In [ ]:
cols_corr = ['pIC50', 'poids_moleculaire', 'logP', 'tpsa', 'donneurs_H',
             'accepteurs_H', 'liaisons_rotatives', 'anneaux_aromatiques']
plt.figure(figsize=(10, 8))
sns.heatmap(df[cols_corr].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, linewidths=0.5, annot_kws={'size': 9})
plt.title("Correlations entre proprietes moleculaires et activite")
plt.tight_layout()
plt.show()

print("La colonne du pIC50 est la plus interessante pour nous. Les correlations avec les descripteurs individuels sont faibles, ce qui confirme le constat precedent : la puissance d'une molecule ne se lit pas dans une seule de ses proprietes. En revanche, on voit des correlations fortes entre descripteurs eux-memes, par exemple entre le poids moleculaire et le nombre d'accepteurs de liaisons hydrogene, ce qui annonce qu'une reduction de dimension sera utile pour resumer cette information redondante.")

# 4. Le graphique signature : l'espace de Lipinski colore par activite

On place chaque molecule dans le plan defini par son poids et son LogP, les deux axes historiques de la druggabilite, et on colore par la puissance d'inhibition. C'est la carte que tout chimiste medicinal a en tete.

In [ ]:
plt.figure(figsize=(14, 8))
sc = plt.scatter(df['poids_moleculaire'], df['logP'], c=df['pIC50'],
                 cmap='viridis', s=28, alpha=0.6, edgecolors='none')
plt.colorbar(sc, label='pIC50 (puissance)')

# On materialise les limites de Lipinski
plt.axvline(500, color=C_ROUGE, linestyle='--', linewidth=1.2, alpha=0.7)
plt.axhline(5, color=C_ROUGE, linestyle='--', linewidth=1.2, alpha=0.7)
plt.text(510, plt.ylim()[0] + 0.3, 'poids = 500', color=C_ROUGE, fontsize=9)
plt.text(plt.xlim()[0] + 10, 5.1, 'LogP = 5', color=C_ROUGE, fontsize=9)

plt.title("L'espace chimique de l'EGFR : poids contre LogP, colore par puissance")
plt.xlabel("Poids moleculaire")
plt.ylabel("LogP (caractere hydrophobe)")
plt.tight_layout()
plt.show()

print("Cette carte resume la situation en une image. Les molecules puissantes, en jaune et vert clair, se concentrent dans une region precise de l'espace, plutot vers un poids moyen a eleve et un LogP modere. Les lignes rouges marquent les limites de Lipinski : on voit que la majorite des molecules interessantes vivent dans le quadrant conforme aux regles de druggabilite. Pour un chercheur, ce graphique dit ou chercher : viser cette zone dense de molecules actives et respectueuses de Lipinski maximise les chances de trouver un bon candidat.")

# Conclusion

L'exploration livre un message clair et un peu contre-intuitif : il n'existe pas de propriete magique qui separe a elle seule les bonnes molecules des mauvaises. L'activite sur l'EGFR est un phenomene multifactoriel, ce qui justifie pleinement le recours a des modeles capables de combiner de nombreuses variables.

On a tout de meme degage une zone privilegiee de l'espace chimique ou se concentrent les molecules actives et druggables. La limite de cette analyse est qu'elle reste bidimensionnelle et lineaire dans son raisonnement. Les modeles des notebooks suivants, et surtout la carte t-SNE construite a partir des empreintes structurelles, iront beaucoup plus loin dans la cartographie fine de cet espace.